# Resume Screening System - Exploration Notebook

This notebook demonstrates the usage of the AI-Powered Resume Screening System.

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data.parser import ResumeParser, JobDescriptionParser
from src.extraction.skill_extractor import SkillExtractor
from src.extraction.experience_extractor import ExperienceExtractor
from src.extraction.entity_extractor import EntityExtractor
from src.models.resume import Resume
from src.models.job import JobDescription
from src.matching.job_matcher import JobMatcher
from src.analysis.gap_analyzer import GapAnalyzer

print("✅ Imports successful!")

## 1. Parse Resume

In [ ]:
# Load sample resume
resume_path = Path("../data/sample_resumes/sample_resume_1.txt")

parser = ResumeParser()
resume_text = parser.parse_file(resume_path)

print("Resume Text (first 500 chars):")
print(resume_text[:500])

## 2. Extract Information

In [ ]:
# Extract entities
entity_extractor = EntityExtractor()
entities = entity_extractor.extract_all_entities(resume_text)

print("\n=== Extracted Entities ===")
print(f"Name: {entities.get('name')}")
print(f"Email: {entities.get('email')}")
print(f"Phone: {entities.get('phone')}")
print(f"\nEducation: {len(entities.get('education', []))} entries found")
print(f"Certifications: {len(entities.get('certifications', []))} found")

In [ ]:
# Extract skills
skill_extractor = SkillExtractor()
skills = skill_extractor.extract_skills(resume_text)

print("\n=== Extracted Skills ===")
print(f"Total skills found: {len(skills)}")
print(f"Skills: {', '.join(skills[:15])}...")

# Categorize skills
categorized = skill_extractor.categorize_skills(skills)
print(f"\nLanguages: {len(categorized['languages'])}")
print(f"Technical: {len(categorized['technical'])}")
print(f"Tools: {len(categorized['tools'])}")

In [ ]:
# Extract experience
experience_extractor = ExperienceExtractor()
experiences = experience_extractor.extract_experience(resume_text)
total_exp = experience_extractor.calculate_total_experience(experiences)

print("\n=== Work Experience ===")
print(f"Total experience: {total_exp} years")
print(f"Number of positions: {len(experiences)}")

for i, exp in enumerate(experiences, 1):
    print(f"\n{i}. {exp.title} at {exp.company}")
    print(f"   Duration: {exp.duration}")

## 3. Parse Job Description

In [ ]:
# Load job description
job_path = Path("../data/sample_jobs/job_description_1.txt")

job_parser = JobDescriptionParser()
job_data = job_parser.parse_file(job_path)

# Extract job skills
with open(job_path, 'r') as f:
    job_text = f.read()

job_skills = skill_extractor.extract_skills(job_text)

print("\n=== Job Requirements ===")
print(f"Skills required: {len(job_skills)}")
print(f"Top skills: {', '.join(job_skills[:10])}")

## 4. Create Models

In [ ]:
# Create Resume object
resume = Resume(
    raw_text=resume_text,
    name=entities.get('name'),
    email=entities.get('email'),
    phone=entities.get('phone'),
    skills=skills,
    education=[],
    experience=experiences,
    total_experience_years=total_exp
)

# Create Job object
job = JobDescription(
    title="Senior Full Stack Engineer",
    company="TechCorp Solutions",
    description=job_text,
    required_skills=job_skills[:10],
    preferred_skills=job_skills[10:15],
    required_experience=5.0
)

print("✅ Models created successfully")

## 5. Match Resume to Job

In [ ]:
# Perform matching
matcher = JobMatcher()
match_result = matcher.match_resume_to_job(resume, job)

print("\n" + "="*60)
print("MATCH RESULTS")
print("="*60)
print(f"Overall Score: {match_result['overall_score']}%")
print(f"Match Level: {match_result['match_level']}")
print(f"\nComponent Scores:")
print(f"  - Skills: {match_result['skill_score']}%")
print(f"  - Experience: {match_result['experience_score']}%")
print(f"  - Education: {match_result['education_score']}%")

print(f"\nMatched Skills ({len(match_result['matched_skills'])}):")
print(f"  {', '.join(match_result['matched_skills'][:10])}")

if match_result['missing_required_skills']:
    print(f"\nMissing Skills ({len(match_result['missing_required_skills'])}):")
    print(f"  {', '.join(match_result['missing_required_skills'])}")

print(f"\nRecommendation: {match_result['recommendation']}")

## 6. Gap Analysis

In [ ]:
# Perform gap analysis
analyzer = GapAnalyzer()
gap_analysis = analyzer.analyze_gaps(resume, job)

print("\n" + "="*60)
print("GAP ANALYSIS")
print("="*60)

skill_gaps = gap_analysis['skill_gaps']
print(f"\nSkill Coverage:")
print(f"  Required: {skill_gaps['required_skills_coverage']}%")
print(f"  Preferred: {skill_gaps['preferred_skills_coverage']}%")

exp_gaps = gap_analysis['experience_gaps']
print(f"\nExperience:")
print(f"  Required: {exp_gaps['required_experience']} years")
print(f"  Candidate: {exp_gaps['candidate_experience']} years")
print(f"  Status: {'✅ Met' if exp_gaps['meets_requirement'] else '❌ Gap'}")

print(f"\nPriority Areas:")
for area in gap_analysis['priority_areas']:
    print(f"  - {area}")

print(f"\nImprovement Suggestions:")
for i, suggestion in enumerate(gap_analysis['improvement_suggestions'][:5], 1):
    print(f"{i}. {suggestion}")

## 7. Visualization (Optional)

In [ ]:
# Note: Uncomment if matplotlib is installed
# import matplotlib.pyplot as plt

# # Create bar chart of scores
# categories = ['Skills', 'Experience', 'Education']
# scores = [
#     match_result['skill_score'],
#     match_result['experience_score'],
#     match_result['education_score']
# ]

# plt.figure(figsize=(10, 6))
# plt.bar(categories, scores, color=['#3498db', '#2ecc71', '#e74c3c'])
# plt.title('Match Score Breakdown', fontsize=16, fontweight='bold')
# plt.ylabel('Score (%)', fontsize=12)
# plt.ylim(0, 100)
# plt.grid(axis='y', alpha=0.3)
# for i, v in enumerate(scores):
#     plt.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
# plt.tight_layout()
# plt.show()

print("Visualization section - uncomment if matplotlib installed")

## Conclusion

This notebook demonstrated:
1. Resume parsing from file
2. Information extraction (skills, experience, entities)
3. Job description parsing
4. Resume-job matching with scoring
5. Gap analysis with recommendations

Feel free to experiment with different resumes and job descriptions!